In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("prince2004patel/iti-student-dropout-synthetic-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'iti-student-dropout-synthetic-dataset' dataset.
Path to dataset files: /kaggle/input/iti-student-dropout-synthetic-dataset


In [ ]:
import os

path = "/root/.cache/kagglehub/datasets/prince2004patel/iti-student-dropout-synthetic-dataset/versions/1"

print(os.listdir(path))

['EDA.ipynb', 'iti_student_dropout_dataset.csv', 'README.md']


In [ ]:
import pandas as pd
import numpy as np

file_path = "/kaggle/input/iti-student-dropout-synthetic-dataset/iti_student_dropout_dataset.csv"

df = pd.read_csv(file_path)

In [ ]:
df.to_csv('output.csv', index=False)

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# === ADJUST THESE KEYS to match actual column names in the CSV ===
column_mapping = {
    'gender':                  'gender',
    'age':                     'age',
    'program_enrolled':        'program_enrolled',       # e.g., "Trade" or "Course"
    'attendance_rate':         'attendance_rate',
    'backlogs':                'backlogs',
    'practical_skills_rating': 'practical_skills_rating',
    'test_scores_avg':             'test_scores',
    'dropout':                 'dropout',
}

# Select only the columns we need
df = df[list(column_mapping.keys())].copy()

# Rename to a clean, consistent format
df.rename(columns=column_mapping, inplace=True)

print("Columns after selection & rename:")
print(df.columns.tolist())
print(df.head())

Columns after selection & rename:
['gender', 'age', 'program_enrolled', 'attendance_rate', 'backlogs', 'practical_skills_rating', 'test_scores', 'dropout']
   gender  age program_enrolled  attendance_rate  backlogs  \
0  Female   21              ECE             77.2         1   
1    Male   21       Electrical             60.8         0   
2    Male   20      Computer IT             57.2         0   
3    Male   20         Plumbing             62.6         1   
4  Female   22              ECE             83.9         2   

   practical_skills_rating  test_scores dropout  
0                     84.3         67.1      No  
1                     73.5         47.5     Yes  
2                     78.2         66.2     Yes  
3                     43.6         44.9     Yes  
4                    100.0        100.0      No  


In [ ]:
print("--- Missing Values Before Cleaning ---")
print(df.isnull().sum())

# 3a. Numerical columns — impute with median (robust to outliers)
numerical_cols = ['age', 'attendance_rate', 'backlogs', 'practical_skills_rating', 'test_scores']
for col in numerical_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f"  Imputed {col} with median: {median_val}")

# 3b. Categorical columns — impute with mode
categorical_cols = ['gender', 'program_enrolled', 'dropout']
for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0]
        df[col].fillna(mode_val, inplace=True)
        print(f"  Imputed {col} with mode: {mode_val}")

print("\n--- Missing Values After Cleaning ---")
print(df.isnull().sum())

--- Missing Values Before Cleaning ---
gender                     0
age                        0
program_enrolled           0
attendance_rate            0
backlogs                   0
practical_skills_rating    0
test_scores                0
dropout                    0
dtype: int64

--- Missing Values After Cleaning ---
gender                     0
age                        0
program_enrolled           0
attendance_rate            0
backlogs                   0
practical_skills_rating    0
test_scores                0
dropout                    0
dtype: int64


In [ ]:
# 4a. Gender — standardize to 'Male' / 'Female' / 'Other'
df['gender'] = df['gender'].astype(str).str.strip().str.title()
valid_genders = ['Male', 'Female', 'Other']
df['gender'] = df['gender'].apply(lambda x: x if x in valid_genders else 'Other')
print("Gender distribution:\n", df['gender'].value_counts())

# 4b. Age — ensure integer, clip to realistic range (14–60)
df['age'] = pd.to_numeric(df['age'], errors='coerce')
df['age'] = df['age'].fillna(df['age'].median())
df['age'] = df['age'].clip(lower=14, upper=60).astype(int)
print(f"\nAge range: {df['age'].min()} – {df['age'].max()}")

# 4c. Program Enrolled — strip whitespace, title case
df['program_enrolled'] = df['program_enrolled'].astype(str).str.strip().str.title()
print(f"\nPrograms: {df['program_enrolled'].unique()}")

# 4d. Attendance Rate — ensure float in range [0, 100]
df['attendance_rate'] = pd.to_numeric(df['attendance_rate'], errors='coerce')
df['attendance_rate'] = df['attendance_rate'].fillna(df['attendance_rate'].median())
df['attendance_rate'] = df['attendance_rate'].clip(lower=0, upper=100).round(2)

# 4e. Backlogs — ensure integer in range [0, 9]
df['backlogs'] = pd.to_numeric(df['backlogs'], errors='coerce')
df['backlogs'] = df['backlogs'].fillna(0)
df['backlogs'] = df['backlogs'].clip(lower=0, upper=9).astype(int)

# 4f. Practical Skills Rating — ensure float in range [0, 10]
df['practical_skills_rating'] = pd.to_numeric(df['practical_skills_rating'], errors='coerce')
df['practical_skills_rating'] = df['practical_skills_rating'].fillna(df['practical_skills_rating'].median())
df['practical_skills_rating'] = df['practical_skills_rating'].clip(lower=0, upper=10).round(2)

# 4g. Test Scores — ensure float in range [0, 100]
df['test_scores'] = pd.to_numeric(df['test_scores'], errors='coerce')
df['test_scores'] = df['test_scores'].fillna(df['test_scores'].median())
df['test_scores'] = df['test_scores'].clip(lower=0, upper=100).round(2)

print("\n--- Data Types After Fixing ---")
print(df.dtypes)
print(df.describe())

Gender distribution:
 gender
Male      4588
Female    2412
Name: count, dtype: int64

Age range: 15 – 30

Programs: ['Ece' 'Electrical' 'Computer It' 'Plumbing' 'Civil' 'Mechanical'
 'Welding']

--- Data Types After Fixing ---
gender                      object
age                          int64
program_enrolled            object
attendance_rate            float64
backlogs                     int64
practical_skills_rating    float64
test_scores                float64
dropout                     object
dtype: object
               age  attendance_rate     backlogs  practical_skills_rating  \
count  7000.000000      7000.000000  7000.000000                   7000.0   
mean     20.326143        58.339714     1.568000                     10.0   
std       2.700363        23.003713     2.103061                      0.0   
min      15.000000        10.000000     0.000000                     10.0   
25%      18.000000        42.375000     0.000000                     10.0   
50%      20.00000

In [ ]:
# 5a. Remove exact duplicate rows
before = len(df)
df.drop_duplicates(inplace=True)
print(f"Removed {before - len(df)} duplicate rows. Remaining: {len(df)}")

# 5b. Outlier detection using IQR method (flag, don't blindly remove)
def flag_outliers_iqr(series, factor=1.5):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - factor * IQR
    upper = Q3 + factor * IQR
    return (series < lower) | (series > upper)

outlier_cols = ['age', 'attendance_rate', 'test_scores', 'practical_skills_rating']
for col in outlier_cols:
    outliers = flag_outliers_iqr(df[col])
    print(f"  {col}: {outliers.sum()} outliers detected")

# Optional: remove rows where multiple columns have outliers simultaneously
# For now, the clipping in Step 4 handles extreme values.

Removed 0 duplicate rows. Remaining: 7000
  age: 14 outliers detected
  attendance_rate: 0 outliers detected
  test_scores: 0 outliers detected
  practical_skills_rating: 0 outliers detected


In [ ]:
df.to_csv('output.csv', index=False)

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
df.head()

,gender,age,program_enrolled,attendance_rate,backlogs,practical_skills_rating,test_scores,dropout
0,Female,21,Ece,77.2,1,10.0,67.1,No
1,Male,21,Electrical,60.8,0,10.0,47.5,Yes
2,Male,20,Computer It,57.2,0,10.0,66.2,Yes
3,Male,20,Plumbing,62.6,1,10.0,44.9,Yes
4,Female,22,Ece,83.9,2,10.0,100.0,No


In [ ]:
# 1) Age: set all ages below 17 to 17
df["age"] = df["age"].clip(lower=17)

# 2) Rename column
df = df.rename(columns={"practical_skills_rating": "assignment_score"})

# 3) Normalize assignment_score to max=15
# (Assumes current values are on a 0–10 scale, like in your sample where values are 10.0)
df["assignment_score"] = (df["assignment_score"] / 10) * 15

# Optional: keep clean rounding
df["assignment_score"] = df["assignment_score"].round(2)

In [ ]:
import pandas as pd
import numpy as np

# Ensure numeric
cols = ["attendance_rate", "test_scores", "assignment_score", "backlogs"]
df[cols] = df[cols].apply(pd.to_numeric, errors="coerce")

def predict_dropout_label(row):
    test = row["test_scores"]          # highest priority (0.5)
    att  = row["attendance_rate"]      # medium priority (0.2)
    assn = row["assignment_score"]     # medium priority (0.2), assumed 0-15 scale
    bl   = row["backlogs"]             # lower priority (0.1)

    # ---------- HIGH risk (dropout = high) ----------
    # Very poor test score OR poor test + other weak indicators
    if (
        test < 40
        or (test < 50 and (att < 60 or assn < 8 or bl >= 5))
        or (test < 55 and att < 50 and assn < 7)
        or (bl >= 8 and test < 60)
    ):
        return "high"

    # ---------- LOW risk (dropout = low) ----------
    # Strong test score, with decent attendance/assignment and low backlog
    if (
        (test >= 75 and att >= 65 and assn >= 9 and bl <= 2)
        or (test >= 85 and bl <= 3)
        or (test >= 70 and att >= 75 and assn >= 10 and bl <= 1)
    ):
        return "low"

    # ---------- MEDIUM risk ----------
    return "medium"

# Apply prediction
df["dropout"] = df.apply(predict_dropout_label, axis=1)

# Check distribution
print(df["dropout"].value_counts())

dropout
medium    4503
high      1418
low       1079
Name: count, dtype: int64


In [ ]:
df.head()

,gender,age,program_enrolled,attendance_rate,backlogs,assignment_score,test_scores,dropout
0,Female,21,Ece,77.2,1,15.0,67.1,medium
1,Male,21,Electrical,60.8,0,15.0,47.5,medium
2,Male,20,Computer It,57.2,0,15.0,66.2,medium
3,Male,20,Plumbing,62.6,1,15.0,44.9,medium
4,Female,22,Ece,83.9,2,15.0,100.0,low


In [ ]:
# Save processed dataframe
df.to_csv("students_processed.csv", index=False)